In [1]:
# Import initial libraries
import pandas as pd
import numpy as np
import boto3
from io import StringIO
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load your modeling dataset from S3
# Replace the path below with your actual S3 path from Day 26
s3 = boto3.client('s3')
bucket_name = 'cmse492-hopki219-nyc311-339713165743-us-east-1-an'
file_name = 'modeling/modeling_data.csv' # make sure to include the path if it's in a folder, e.g. 'modeling/modeling_data.csv'

obj = s3.get_object(Bucket=bucket_name, Key=file_name)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data))

print(f"Shape: {df.shape}")
df.head()

Shape: (173851, 8)


,agency,borough,problem,incident_zip,day_of_week,hour_of_day,same_day_complaint_volume,days_to_close
0,DCWP,BROOKLYN,Consumer Complaint,NaN,4,12,24,0
1,DCWP,QUEENS,Consumer Complaint,11040.0,4,13,24,0
2,DCWP,BROOKLYN,Consumer Complaint,11212.0,4,8,24,36
3,DCWP,MANHATTAN,Consumer Complaint,10018.0,4,12,24,35
4,DCWP,BROOKLYN,Consumer Complaint,11236.0,4,12,24,30


In [79]:
df = df.dropna()

In [80]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

Columns: ['agency', 'borough', 'problem', 'incident_zip', 'day_of_week', 'hour_of_day', 'same_day_complaint_volume', 'days_to_close']

Missing values:
agency                       0
borough                      0
problem                      0
incident_zip                 0
day_of_week                  0
hour_of_day                  0
same_day_complaint_volume    0
days_to_close                0
dtype: int64

Data types:
agency                        object
borough                       object
problem                       object
incident_zip                 float64
day_of_week                    int64
hour_of_day                    int64
same_day_complaint_volume      int64
days_to_close                  int64
dtype: object


In [81]:
# Define feature columns and target column
# Replace these with your actual column names
feature_cols = ['agency', 'problem', 'incident_zip', 'day_of_week', 'hour_of_day', 'same_day_complaint_volume']  # <-- update this
target_col   = 'days_to_close'          # <-- update this,

# If your target needs to be binarized or modified in some way, do it here
# Example: df['high_volume'] = (df['volume_quartile'] == 1).astype(int)

X = df[feature_cols]
y = df[target_col]

print(f"Features: {feature_cols}")
print(f"Target: {target_col}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}")

Features: ['agency', 'problem', 'incident_zip', 'day_of_week', 'hour_of_day', 'same_day_complaint_volume']
Target: days_to_close

Target distribution:
days_to_close
0     106992
1      17454
2      12225
3       8085
4       5758
5       4518
6       3094
7       1909
8       1396
9       1229
10       819
11       722
12       677
13       606
14       551
15       490
16       444
17       430
19       391
18       375
20       321
21       281
22       271
23       264
26       230
30       225
25       224
24       214
27       186
29       170
28       167
32       161
31       153
33       137
35       106
36        98
34        97
40        85
38        78
37        76
41        70
39        69
42        53
45        38
43        37
44        34
46        29
48        19
47        16
49        12
Name: count, dtype: int64

Class balance: {0: 0.622, 1: 0.101, 2: 0.071, 3: 0.047, 4: 0.033, 5: 0.026, 6: 0.018, 7: 0.011, 8: 0.008, 9: 0.007, 10: 0.005, 11: 0.004, 12: 0.004, 13: 0.004

In [82]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Identify categorical vs. numeric columns automatically
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numeric columns     ({len(num_cols)}): {num_cols}")

# Build the transformer:
#   - OneHotEncoder for categorical columns
#     drop='first'         avoids multicollinearity (k-1 dummies per feature)
#     sparse_output=False  returns a regular numpy array (easier to inspect)
#     handle_unknown='ignore'  silently zeros out categories not seen in training
#   - StandardScaler for numeric columns (optional but often helpful for models like Logistic Regression)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols),  # scale numeric columns
    ]
)

# Fit ONLY on training data, then apply to both splits
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

# Recover feature names for later coefficient inspection
ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
encoded_feature_names = ohe_names + num_cols

print(f"\nEncoded feature matrix shape: {X_train_enc.shape}")
print(f"Total features after encoding: {len(encoded_feature_names)}")
print(f"\nFirst few encoded feature names: {encoded_feature_names[:10]}")

Categorical columns (2): ['agency', 'problem']
Numeric columns     (4): ['incident_zip', 'day_of_week', 'hour_of_day', 'same_day_complaint_volume']

Encoded feature matrix shape: (137668, 156)
Total features after encoding: 156

First few encoded feature names: ['agency_DEP', 'agency_DHS', 'agency_DOB', 'agency_DOE', 'agency_DOHMH', 'agency_DOT', 'agency_DPR', 'agency_DSNY', 'agency_HPD', 'agency_NYPD']


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [83]:
# Import train_test_split for splitting the data
from sklearn.model_selection import train_test_split

# Train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).round(3))

Training set:  137668 rows
Test set:      34418 rows

Class balance in training set:
days_to_close
0     0.622
1     0.101
2     0.071
3     0.047
4     0.033
5     0.026
6     0.018
7     0.011
8     0.008
9     0.007
10    0.005
11    0.004
12    0.004
13    0.004
14    0.003
15    0.003
16    0.003
17    0.002
19    0.002
18    0.002
20    0.002
21    0.002
22    0.002
23    0.002
26    0.001
30    0.001
25    0.001
24    0.001
27    0.001
29    0.001
28    0.001
32    0.001
31    0.001
33    0.001
35    0.001
36    0.001
34    0.001
40    0.000
38    0.000
37    0.000
41    0.000
39    0.000
42    0.000
43    0.000
45    0.000
44    0.000
46    0.000
48    0.000
47    0.000
49    0.000
Name: proportion, dtype: float64

Class balance in test set:
days_to_close
0     0.622
1     0.101
2     0.071
3     0.047
4     0.033
5     0.026
6     0.018
7     0.011
8     0.008
9     0.007
10    0.005
11    0.004
12    0.004
13    0.004
14    0.003
15    0.003
16    0.003
17    0.002
19    0.00

In [99]:
# Import the logistic regression model
from sklearn.linear_model import LinearRegression

# Train the logistic regression baseline (adjust the model type and parameters as needed for your specific task)
model = LinearRegression()
model.fit(X_train_enc, y_train)

print("Model trained successfully.")
#print(f"Classes: {model.classes_}")

Model trained successfully.


In [100]:
# Inspect the model coefficients
# After encoding, use encoded_feature_names instead of feature_cols
coef_df = pd.DataFrame({
    'feature': encoded_feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Model coefficients (top 10 by magnitude):")
print(coef_df.head(10).to_string(index=False))
print("\nPositive coefficient = feature pushes toward the positive class")
print("Negative coefficient = feature pushes toward the negative class")

Model coefficients (top 10 by magnitude):
                               feature  coefficient
             same_day_complaint_volume    -4.240116
                            agency_DEP    -4.240116
                            agency_DHS    -4.240116
                            agency_DOB    -4.240116
problem_Unsanitary Animal Pvt Property    -4.240116
   problem_Unsanitary Pigeon Condition    -4.240116
                problem_Uprooted Stump    -4.240116
           problem_Urinating in Public    -4.240116
            problem_Vendor Enforcement    -4.240116
       problem_Violation of Park Rules    -4.240116

Positive coefficient = feature pushes toward the positive class
Negative coefficient = feature pushes toward the negative class


In [103]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Generate predictions on the test set
y_pred = model.predict(X_test_enc)


In [109]:
from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
mape

2.3832272290551555e+17